In [1]:
import os
os.chdir(r"C:\Users\Lenovo\Desktop\KYC_Project")
print("Working directory:", os.getcwd())

Working directory: C:\Users\Lenovo\Desktop\KYC_Project


In [2]:
def kyc_decision(
    ocr_result,
    face_result,
    liveness_result,
    stamp_result,
    tampering_result
):
    """
    Combines all module results into final KYC decision.
    
    Each result dict must have a score field:
      ocr_result       → overall_match (True/False)
      face_result      → score (0.0-1.0)
      liveness_result  → score (0.0-1.0)
      stamp_result     → avg_score (0.0-1.0)
      tampering_result → genuine_score (0.0-1.0)
    """

    # ── Extract scores ───────────────────────────────
    ocr_score    = 1.0 if ocr_result.get("overall_match") else 0.0
    face_score   = face_result.get("score", 0.0)
    live_score   = liveness_result.get("score", 0.0)
    stamp_score  = stamp_result.get("avg_score", 0.0)
    tamper_score = tampering_result.get("genuine_score", 0.0)

    # ── Hard rules (instant reject) ──────────────────
    hard_reject_reason = None

    if not liveness_result.get("is_live"):
        hard_reject_reason = "liveness check failed — possible spoof attempt"

    if not tampering_result.get("is_genuine"):
        hard_reject_reason = "tampering detected — document may be forged"

    # ── Weighted score ───────────────────────────────
    weighted_score = (
        ocr_score    * 0.25 +
        face_score   * 0.25 +
        live_score   * 0.20 +
        stamp_score  * 0.15 +
        tamper_score * 0.15
    )

    # ── Final decision ───────────────────────────────
    if hard_reject_reason:
        status = "REJECTED"
        reason = hard_reject_reason
    elif weighted_score >= 0.6:
        status = "APPROVED"
        reason = "all checks passed"
    else:
        status = "REJECTED"
        reason = "insufficient verification score"

    return {
        "status": status,
        "weighted_score": round(weighted_score, 4),
        "reason": reason,
        "breakdown": {
            "ocr":       {"score": ocr_score,    "weight": "25%"},
            "face":      {"score": face_score,   "weight": "25%"},
            "liveness":  {"score": live_score,   "weight": "20%"},
            "stamp":     {"score": stamp_score,  "weight": "15%"},
            "tampering": {"score": tamper_score, "weight": "15%"},
        }
    }

print("kyc_decision function ready")

kyc_decision function ready


In [3]:
def print_result(label, result):
    print(f"\n{'='*50}")
    print(f"Test: {label}")
    print(f"  Status:         {result['status']}")
    print(f"  Weighted Score: {result['weighted_score']}")
    print(f"  Reason:         {result['reason']}")
    print(f"  Breakdown:")
    for module, data in result["breakdown"].items():
        print(f"    {module:<12} score={data['score']}  weight={data['weight']}")

# Test 1 — All pass (should APPROVE)
result1 = kyc_decision(
    ocr_result       = {"overall_match": True},
    face_result      = {"score": 0.85, "match": True},
    liveness_result  = {"score": 0.82, "is_live": True},
    stamp_result     = {"avg_score": 0.75, "is_genuine": True},
    tampering_result = {"genuine_score": 0.77, "is_genuine": True},
)
print_result("All checks pass", result1)

# Test 2 — Liveness fails (should REJECT)
result2 = kyc_decision(
    ocr_result       = {"overall_match": True},
    face_result      = {"score": 0.85, "match": True},
    liveness_result  = {"score": 0.12, "is_live": False},
    stamp_result     = {"avg_score": 0.75, "is_genuine": True},
    tampering_result = {"genuine_score": 0.77, "is_genuine": True},
)
print_result("Liveness fails", result2)

# Test 3 — Tampering detected (should REJECT)
result3 = kyc_decision(
    ocr_result       = {"overall_match": True},
    face_result      = {"score": 0.85, "match": True},
    liveness_result  = {"score": 0.82, "is_live": True},
    stamp_result     = {"avg_score": 0.75, "is_genuine": True},
    tampering_result = {"genuine_score": 0.22, "is_genuine": False},
)
print_result("Tampering detected", result3)

# Test 4 — OCR and face both weak (should REJECT)
result4 = kyc_decision(
    ocr_result       = {"overall_match": False},
    face_result      = {"score": 0.25, "match": False},
    liveness_result  = {"score": 0.82, "is_live": True},
    stamp_result     = {"avg_score": 0.75, "is_genuine": True},
    tampering_result = {"genuine_score": 0.77, "is_genuine": True},
)
print_result("OCR and face weak", result4)


Test: All checks pass
  Status:         APPROVED
  Weighted Score: 0.8545
  Reason:         all checks passed
  Breakdown:
    ocr          score=1.0  weight=25%
    face         score=0.85  weight=25%
    liveness     score=0.82  weight=20%
    stamp        score=0.75  weight=15%
    tampering    score=0.77  weight=15%

Test: Liveness fails
  Status:         REJECTED
  Weighted Score: 0.7145
  Reason:         liveness check failed — possible spoof attempt
  Breakdown:
    ocr          score=1.0  weight=25%
    face         score=0.85  weight=25%
    liveness     score=0.12  weight=20%
    stamp        score=0.75  weight=15%
    tampering    score=0.77  weight=15%

Test: Tampering detected
  Status:         REJECTED
  Weighted Score: 0.772
  Reason:         tampering detected — document may be forged
  Breakdown:
    ocr          score=1.0  weight=25%
    face         score=0.85  weight=25%
    liveness     score=0.82  weight=20%
    stamp        score=0.75  weight=15%
    tampering  

In [4]:
os.makedirs("ml/aggregator", exist_ok=True)

with open("ml/aggregator/aggregator_summary.txt", "w", encoding="utf-8") as f:
    f.write("""
KYC Decision Aggregator
------------------------
Weights:
  OCR matching     25%
  Face matching    25%
  Liveness         20%
  Stamp            15%
  Tampering        15%

Threshold: weighted_score >= 0.6 -> APPROVED

Hard reject rules (override score):
  - Liveness failed  -> REJECTED
  - Tampering found  -> REJECTED

Status values: APPROVED / REJECTED
""")

print("Aggregator saved")


Aggregator saved
